# CFPB Complaints — Logistic Regression on TF-IDF

Traditional-ML baseline for 10-class product classification, following `ML_model_building.md`.
Starts from the already-filtered `cfpb_filtered.parquet` (407,321 rows).

This notebook writes `train.parquet` / `test.parquet` so DistilBERT can be scored on the
*exact same* test set.

In [ ]:
import os
import re
import time
import warnings

import joblib
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
PARQUET = "cfpb_filtered.parquet"

pd.set_option("display.max_colwidth", 60)
print("pandas", pd.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)

## Step 1 — Load the filtered data

Reads the parquet, keeps only `narrative` (text) and `product` (label).
The checkpoint asserts 407,321 rows across 10 classes and stops the notebook if it differs.

In [2]:
df = pd.read_parquet(PARQUET, columns=["narrative", "product"])

print("shape:", df.shape)
print()
class_counts = df["product"].value_counts()
print(class_counts.to_string())

# Checkpoint from the instructions — stop if the data is not what we expect.
EXPECTED_ROWS, EXPECTED_CLASSES = 407_321, 10
assert len(df) == EXPECTED_ROWS, f"expected {EXPECTED_ROWS:,} rows, got {len(df):,}"
assert df["product"].nunique() == EXPECTED_CLASSES, (
    f"expected {EXPECTED_CLASSES} classes, got {df['product'].nunique()}"
)
print(f"\nCheckpoint OK: {len(df):,} rows, {df['product'].nunique()} classes")
print(f"imbalance ratio (max/min): {class_counts.max() / class_counts.min():.1f}x")

shape: (407321, 2)

product
Debt collection                                            134953
Money transfer, virtual currency, or money service          72921
Checking or savings account                                 69343
Credit card                                                 58916
Mortgage                                                    20497
Vehicle loan or lease                                       16712
Student loan                                                14163
Payday loan, title loan, personal loan, or advance loan     11271
Prepaid card                                                 5014
Debt or credit management                                    3531

Checkpoint OK: 407,321 rows, 10 classes
imbalance ratio (max/min): 38.2x


## Step 2 — Balanced sampling

Up to 8,000 rows per class: classes above the cap are randomly sampled down to exactly
8,000, classes below it are taken in full. Deliberately *not* perfectly balanced — the two
small classes stay small, and `class_weight='balanced'` in Step 5 absorbs the remainder.
No oversampling.

In [3]:
CAP = 8_000

# Explicit loop rather than groupby(...).apply(...): as of pandas 3.0, apply excludes the
# grouping column from each group, which silently drops "product" from the result.
parts = [
    g.sample(n=min(len(g), CAP), random_state=RANDOM_STATE)
    for _, g in df.groupby("product", sort=False)
]
balanced = pd.concat(parts, ignore_index=True)

bal_counts = balanced["product"].value_counts()
print(bal_counts.to_string())
print(f"\ntotal rows after balanced sampling: {len(balanced):,}")
print(f"classes at the {CAP:,} cap        : {(bal_counts == CAP).sum()}")
print(f"classes taken in full            : {(bal_counts < CAP).sum()}")

product
Student loan                                               8000
Debt collection                                            8000
Checking or savings account                                8000
Credit card                                                8000
Payday loan, title loan, personal loan, or advance loan    8000
Mortgage                                                   8000
Money transfer, virtual currency, or money service         8000
Vehicle loan or lease                                      8000
Prepaid card                                               5014
Debt or credit management                                  3531

total rows after balanced sampling: 72,545
classes at the 8,000 cap        : 8
classes taken in full            : 2


## Step 3 — Stratified train/test split, saved to disk

80/20, stratified on the label. **Saving the split is the point of this step** — DistilBERT
must be evaluated on this identical test set or the comparison is meaningless.

In [4]:
X = balanced["narrative"]
y = balanced["product"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)

train_df = pd.DataFrame({"narrative": X_train, "product": y_train}).reset_index(drop=True)
test_df = pd.DataFrame({"narrative": X_test, "product": y_test}).reset_index(drop=True)

train_df.to_parquet("train.parquet", index=False)
test_df.to_parquet("test.parquet", index=False)

print(f"train : {len(train_df):,} rows, {train_df['product'].nunique()} classes")
print(f"test  : {len(test_df):,} rows, {test_df['product'].nunique()} classes")
assert train_df["product"].nunique() == 10 and test_df["product"].nunique() == 10, \
    "a class is missing from one of the splits"
print("\nwrote train.parquet and test.parquet  <- shared with the DistilBERT notebook")
print("\ntest-set class balance:")
print(test_df["product"].value_counts().to_string())

train : 58,036 rows, 10 classes
test  : 14,509 rows, 10 classes

wrote train.parquet and test.parquet  <- shared with the DistilBERT notebook

test-set class balance:
product
Mortgage                                                   1600
Vehicle loan or lease                                      1600
Credit card                                                1600
Checking or savings account                                1600
Student loan                                               1600
Payday loan, title loan, personal loan, or advance loan    1600
Money transfer, virtual currency, or money service         1600
Debt collection                                            1600
Prepaid card                                               1003
Debt or credit management                                   706


## Step 4 — Cleaning + TF-IDF

The redaction patterns below were taken from the actual file, not assumed.

So masks are stripped **before** lowercasing (the `X`s are uppercase), longest patterns
first, then leftover `X` runs. Then lowercase, then drop non-alphabetic characters.

TF-IDF is **fit on train only** and merely applied to test — fitting on test would leak.

In [5]:
# Order matters: longest/most specific mask patterns first, while still uppercase.
DATE_MASK = re.compile(r"X{2}/X{2}/(?:X+|\d{2,4}|year>|scrub>)?")
X_RUN = re.compile(r"X{2,}")
NON_ALPHA = re.compile(r"[^a-z\s]")
WS = re.compile(r"\s+")


def clean_text(text: str) -> str:
    """Strip CFPB redaction masks, lowercase, and drop non-alphabetic noise."""
    text = DATE_MASK.sub(" ", text)   # XX/XX/XXXX, XX/XX/year>, XX/XX/2025, XX/XX/
    text = X_RUN.sub(" ", text)       # leftover XXXX / XXXXXXXX runs
    text = text.lower()
    text = NON_ALPHA.sub(" ", text)   # punctuation, digits, stray symbols
    return WS.sub(" ", text).strip()


# Sanity-check the cleaner on a real narrative containing masks.
sample = train_df.loc[train_df["narrative"].str.contains("XX/XX/", regex=False), "narrative"].iloc[0]
print("BEFORE:", sample[:220].replace("\n", " "))
print()
print("AFTER :", clean_text(sample)[:220])

BEFORE: On XX/XX/2020 I visited BMW XXXX XXXXXXXX to get my car serviced. While waiting I was admiring the cars in the showroom when approached by a salesman. I told him I wasn't in the market for a new car because I could not a

AFTER : on i visited bmw to get my car serviced while waiting i was admiring the cars in the showroom when approached by a salesman i told him i wasn t in the market for a new car because i could not afford it he asked me how mu


In [6]:
X_train_clean = X_train.map(clean_text)
X_test_clean = X_test.map(clean_text)

vectorizer = TfidfVectorizer(
    max_features=20_000,
    stop_words="english",
    ngram_range=(1, 2),  # unigrams + bigrams; drop to (1, 1) if memory is tight
)

X_train_tfidf = vectorizer.fit_transform(X_train_clean)  # FIT on train only
X_test_tfidf = vectorizer.transform(X_test_clean)        # transform test with train's vocab

print("TF-IDF train matrix:", X_train_tfidf.shape)
print("TF-IDF test  matrix:", X_test_tfidf.shape)
print(f"vocabulary size    : {len(vectorizer.vocabulary_):,}")
density = X_train_tfidf.nnz / (X_train_tfidf.shape[0] * X_train_tfidf.shape[1])
print(f"sparse density     : {density:.4%}  ({X_train_tfidf.nnz:,} stored values)")

TF-IDF train matrix: (58036, 20000)
TF-IDF test  matrix: (14509, 20000)
vocabulary size    : 20,000
sparse density     : 0.4482%  (5,202,406 stored values)


## Step 5 — Train Logistic Regression

`class_weight='balanced'` is the imbalance handling and is required. If the fit emits a
`ConvergenceWarning` it is refit at `max_iter=2000` rather than ignored.

Note on `n_jobs=-1`: it is set as the instructions specify, but scikit-learn's default
`lbfgs` solver on a multinomial problem ignores it — it only takes effect for one-vs-rest
solvers. It is harmless, not a speedup here.

In [ ]:
def fit_lr(max_iter: int):
    clf = LogisticRegression(
        class_weight="balanced",
        max_iter=max_iter,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", ConvergenceWarning)
        clf.fit(X_train_tfidf, y_train)
    converged = not any(issubclass(w.category, ConvergenceWarning) for w in caught)
    return clf, converged


t0 = time.time()
clf, converged = fit_lr(1000)
print(f"fit with max_iter=1000 in {time.time() - t0:.1f}s  (converged: {converged})")

if not converged:
    print("ConvergenceWarning raised -> refitting at max_iter=2000 as instructed")
    t0 = time.time()
    clf, converged = fit_lr(2000)
    print(f"fit with max_iter=2000 in {time.time() - t0:.1f}s  (converged: {converged})")
    if not converged:
        print("STILL not converged — metrics below are usable but consider max_iter=4000")

print(f"\nclasses: {len(clf.classes_)}")
print(f"coefficient matrix: {clf.coef_.shape}")

## Step 6 — Evaluate on the held-out test set

Headline metric is **macro-averaged F1**, which weights every class equally and so exposes
weak performance on the rare classes. Accuracy alone is misleading at 38x imbalance.

In [8]:
y_pred = clf.predict(X_test_tfidf)

labels = sorted(y_test.unique())
report_txt = classification_report(y_test, y_pred, labels=labels, digits=3, zero_division=0)
print(report_txt)

                                                         precision    recall  f1-score   support

                            Checking or savings account      0.763     0.794     0.779      1600
                                            Credit card      0.815     0.772     0.793      1600
                                        Debt collection      0.782     0.809     0.795      1600
                              Debt or credit management      0.482     0.582     0.528       706
     Money transfer, virtual currency, or money service      0.875     0.815     0.844      1600
                                               Mortgage      0.941     0.899     0.919      1600
Payday loan, title loan, personal loan, or advance loan      0.757     0.763     0.760      1600
                                           Prepaid card      0.711     0.802     0.754      1003
                                           Student loan      0.962     0.911     0.936      1600
                             

In [9]:
macro_f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)
weighted_f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)
accuracy = accuracy_score(y_test, y_pred)

print(f"HEADLINE  macro F1 : {macro_f1:.4f}")
print(f"          weighted F1: {weighted_f1:.4f}")
print(f"          accuracy   : {accuracy:.4f}")
print()
print(f"accuracy - macro F1 gap: {accuracy - macro_f1:+.4f}"
      "  (a positive gap means the rare classes lag the common ones)")

HEADLINE  macro F1 : 0.7973
          weighted F1: 0.8158
          accuracy   : 0.8132

accuracy - macro F1 gap: +0.0158  (a positive gap means the rare classes lag the common ones)


In [10]:
# Short axis labels — the real class names are far too long for a readable plot.
SHORT = {
    "Debt collection": "Debt collection",
    "Money transfer, virtual currency, or money service": "Money transfer/crypto",
    "Checking or savings account": "Checking/savings",
    "Credit card": "Credit card",
    "Mortgage": "Mortgage",
    "Vehicle loan or lease": "Vehicle loan",
    "Student loan": "Student loan",
    "Payday loan, title loan, personal loan, or advance loan": "Payday/personal loan",
    "Prepaid card": "Prepaid card",
    "Debt or credit management": "Debt/credit mgmt",
}
short_labels = [SHORT.get(c, c) for c in labels]

cm = confusion_matrix(y_test, y_pred, labels=labels)

fig, ax = plt.subplots(figsize=(11, 9))
ConfusionMatrixDisplay(cm, display_labels=short_labels).plot(
    ax=ax, cmap="Blues", values_format="d", colorbar=True, xticks_rotation=45
)
plt.setp(ax.get_xticklabels(), ha="right", rotation_mode="anchor")
ax.set_title(f"Logistic Regression + TF-IDF — confusion matrix (macro F1 = {macro_f1:.3f})")
fig.tight_layout()
fig.savefig("confusion_matrix_lr.png", dpi=150, bbox_inches="tight")
print("saved confusion_matrix_lr.png")
plt.show()

saved confusion_matrix_lr.png


/var/folders/xv/__dg3mss3f1d57y41qhwz0dh0000gn/T/ipykernel_3564/2518352924.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Interpretation — strongest/weakest classes and the top confusions

Computed rather than eyeballed, so the numbers quoted in the report's Business Insights
section come straight from the matrix.

In [11]:
per_class_f1 = pd.Series(
    f1_score(y_test, y_pred, labels=labels, average=None, zero_division=0), index=labels
).sort_values(ascending=False)

print("per-class F1 (best -> worst):")
print(per_class_f1.round(3).to_string())
print()
print(f"BEST : {per_class_f1.index[0]!r}  F1={per_class_f1.iloc[0]:.3f}")
print(f"WORST: {per_class_f1.index[-1]!r}  F1={per_class_f1.iloc[-1]:.3f}")

# Most-confused ordered pairs: off-diagonal cells as a share of the true class's row.
cm_df = pd.DataFrame(cm, index=labels, columns=labels)
pairs = [
    {"true": t, "predicted": p, "n": int(cm_df.loc[t, p]),
     "pct_of_true_class": 100 * cm_df.loc[t, p] / cm_df.loc[t].sum()}
    for t in labels for p in labels if t != p
]
top_conf = (pd.DataFrame(pairs)
            .sort_values("pct_of_true_class", ascending=False)
            .head(10)
            .reset_index(drop=True))
top_conf["pct_of_true_class"] = top_conf["pct_of_true_class"].round(1)

print("\ntop 10 confusions (share of the true class sent to the wrong label):")
print(top_conf.to_string(index=False))

per-class F1 (best -> worst):
Student loan                                               0.936
Mortgage                                                   0.919
Vehicle loan or lease                                      0.866
Money transfer, virtual currency, or money service         0.844
Debt collection                                            0.795
Credit card                                                0.793
Checking or savings account                                0.779
Payday loan, title loan, personal loan, or advance loan    0.760
Prepaid card                                               0.754
Debt or credit management                                  0.528

BEST : 'Student loan'  F1=0.936
WORST: 'Debt or credit management'  F1=0.528

top 10 confusions (share of the true class sent to the wrong label):
                                              true                                               predicted   n  pct_of_true_class
                         Debt or credit ma